# p22_agentic_rag

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/javanmardi298/rag_ollama/blob/main/lessons/p22_agentic_rag/p22_agentic_rag.ipynb)

Run the cells in order. In each step: read, complete the TODOs in the code cell, run it, then run the check cell.

In [ ]:
# Setup (Colab: clones the repo; locally: skip if you are already inside it)
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', 'https://github.com/javanmardi298/rag_ollama.git', 'repo'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'repo/requirements.txt', '-e', 'repo'], check=True)
if os.path.exists('repo'): os.chdir('repo')
os.environ.setdefault('LEARN_PROVIDER', 'offline')   # offline | ollama | anthropic | opencode | openai_compatible


# گام ۱: مسیریابی — همه‌ی سؤال‌ها بازیابی نمی‌خواهند

## هدف این گام
تابعی می‌نویسی که برای هر سؤال تصمیم می‌گیرد: **مستقیم جواب بده** (سلام، تشکر، حساب ساده)، **یک بار جست‌وجو کن**، یا **چندمرحله‌ای** (سؤال مرکب). این ایده‌ی Adaptive-RAG است.

## پیش‌نیاز
هیچ؛ فقط رشته و عبارت‌های باقاعده (`re`).

## مفهوم
هر جست‌وجو هزینه دارد (زمان، توکن، پول) و هر جست‌وجوی غیرضروری ریسک هم دارد: برای «hello» تکه‌ای پیدا می‌شود و مدل از آن جواب می‌سازد! مسیریابی سه خروجی دارد:

| مسیر | مثال | رفتار |
|---|---|---|
| `direct` | «hello»، «thanks!»، «2+2» | بدون جست‌وجو جواب بده |
| `retrieve` | «how long do I have to return a sofa» | یک جست‌وجو |
| `multi_hop` | «how long to return and when do I get my money back» | چند زیرسؤال: باید چند بار جست‌وجو کرد |

در سیستم واقعی مسیریاب اغلب یک مدل کوچک یا یک classifier است (در گام ۵ نسخه‌ی مدل زبانی‌اش را می‌بینی). اینجا با **قاعده** شروع می‌کنیم چون سریع، رایگان، قابل‌تست و مثل یک baseline عمل می‌کند: اگر قاعده‌ی ساده کافی است، چرا مدل صدا بزنیم؟

قاعده‌ها:
1. متن را lowercase کن و فاصله‌ها و نشانه‌های پایانی `!.?` ابتدا و انتها را بردار.
2. اگر دقیقاً یکی از `GREETINGS` بود ← `direct`.
3. اگر فقط عدد و عملگر `+-*/().` و فاصله بود ← `direct` (حساب).
4. اگر `" and "` داشت و حداقل دو کلمه‌ی پرسشی (`QWORDS`) در آن بود ← `multi_hop`.
5. وگرنه `retrieve`.

## تمرین کد
فایل پاسخ: `work/p22_agentic_rag/step01_router/answer.py`. تابع `route(question)` را کامل کن.

## اجرا کن
```bash
python -m learn check p22_agentic_rag 1
```

## خروجی مورد انتظار
شش ✓.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| «Hello!» مستقیم نشد | نشانه‌ی `!` را برنداشتی (`rstrip("!.? ")`) |
| «return and refund» چندمرحله‌ای شد | شرط «دو کلمه‌ی پرسشی» را نگذاشتی: `and` به‌تنهایی کافی نیست |
| «what is 2+2» مستقیم شد | حساب باید کل رشته باشد (`re.fullmatch`) نه بخشی از آن |

## دام‌ها
- مسیریاب خطا می‌کند و خطایش **بی‌صدا** است: سؤال سختی که `direct` شود بدون شاهد جواب می‌گیرد. خطای مسیریاب را جدا بسنج (چند درصد سؤال‌ها به مسیر غلط رفتند).
- در تردید `retrieve` بهتر از `direct` است (شکست امن‌تر).

## گام بعد
[گام ۲: نمره‌دادن به شواهد](#)


In [ ]:
"""Step 1: route a question: direct | retrieve | multi_hop."""
from __future__ import annotations

import re

GREETINGS = {"hello", "hi", "hey", "thanks", "thank you", "bye"}
QWORDS = {"how", "what", "when", "where", "why", "which", "who", "can", "do", "does", "is", "are"}


def route(question: str) -> str:
    # TODO 1: normalise (lowercase, strip spaces and trailing "!.?")
    # TODO 2: greeting -> "direct"; pure arithmetic (digits, spaces, + - * / ( ) .) -> "direct"
    # TODO 3: contains " and " and at least two question words -> "multi_hop"; otherwise "retrieve"
    raise NotImplementedError("TODO 1-3: route")


In [ ]:
from learn.notebook import check_cell
check_cell('p22_agentic_rag', 'step01_router', globals())

# گام ۲: نمره‌دادن به شواهد (grading)

## هدف این گام
بعد از جست‌وجو، هر تکه را بررسی کن: «این تکه واقعاً به سؤال مربوط است؟». تکه‌های نامرتبط را دور بریز. این قلب **Corrective RAG** است.

## پیش‌نیاز
گام ۱ مفهومی است؛ این گام مستقل کار می‌کند.

## مفهوم
جست‌وجو همیشه k نتیجه برمی‌گرداند، حتی وقتی هیچ‌کدام درست نیست. سؤال «what is the CEO's phone number» با BM25 تکه‌ی «Send a photo of the defect and your **order number**» را پیدا می‌کند چون کلمه‌ی `number` در آن است. بدون نمره‌دهی، این تکه به مدل می‌رسد و جواب غلط ساخته می‌شود.

**Grader** تصمیم دودویی می‌گیرد: مرتبط / نامرتبط. دو نوع:
- **قاعده‌ای** (این گام): همپوشانی کلمه‌های محتوایی. ارزان، قطعی، قابل‌تست.
- **مدل زبانی** (گام ۵): «آیا این متن به پاسخ کمک می‌کند؟»

قاعده‌ی ما:
1. `content_words(text)`: lowercase، فقط حرف/عدد، حذف `STOP`، و ساده‌ترین ریشه‌یابی: اگر کلمه بیش از ۳ حرف داشت و به `s` ختم می‌شد، `s` را بردار (`invoices` ← `invoice`). مجموعه (set) برگردان.
2. `grade(question, chunk, min_ratio=0.5)`: کلمه‌های محتوایی سؤال (بعد از `expand` که مترادف‌ها را باز می‌کند) و کلمه‌های تکه (عنوان + متن) را بگیر. مرتبط است اگر **حداقل نصف کلمه‌های محتوایی سؤال** در تکه باشد: `len(q & c) / len(q) >= min_ratio`.

**چرا نسبت و نه تعداد؟** «شماره‌ی تلفن مدیرعامل» (`ceo, phone, number`) فقط ۱ از ۳ کلمه را با تکه‌ی «order number» مشترک دارد (۰.۳۳ ← رد). ولی «return a sofa» هم ۱ از ۲ دارد (۰.۵ ← قبول). یک آستانه‌ی ثابت مثل «حداقل ۲ کلمه» سؤال تک‌کلمه‌ای («where is my invoice») را همیشه رد می‌کرد؛ نسبت با طول سؤال تنظیم می‌شود.

`expand` آماده است: عبارت‌هایی مثل `send back` را به `return` تبدیل می‌کند تا هم‌معنی‌ها (محدود) پیدا شوند.

## تمرین کد
فایل پاسخ: `work/p22_agentic_rag/step02_grader/answer.py`. دو تابع `content_words` و `grade` را کامل کن.

## اجرا کن
```bash
python -m learn check p22_agentic_rag 2
```
و برای دیدن رفتار روی داده‌ی واقعی `python work/p22_agentic_rag/step02_grader/answer.py` را اجرا کن: برای هر سؤال، چند تکه‌ی مرتبط می‌ماند.

## خروجی مورد انتظار
هفت ✓. در اجرای دستی، سؤال‌های بی‌جواب (cheque، CEO، gift cards) باید `kept=[]` باشند. برای سؤال‌های قابل‌پاسخ، تکه‌ی درست باید در `kept` باشد (گاهی چند تکه‌ی هم‌خانواده هم می‌ماند؛ طبیعی است).

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| `invoices` و `invoice` یکی نشدند | ریشه‌یابی `s` را نگذاشتی |
| سؤال تک‌کلمه‌ای همیشه رد می‌شود | آستانه را تعداد ثابت گذاشته‌ای نه نسبت |
| CEO تکه‌ی «order number» را گرفت | `min_ratio` را خیلی کم گذاشته‌ای یا `STOP` را اعمال نکرده‌ای |

## دام‌ها
- همپوشانی کلمه **معنا** نیست: «bank transfer» و «return transfer» هم‌پوشان‌اند ولی بی‌ربط. grader قاعده‌ای baseline است؛ در گام ۵ نسخه‌ی مدل را کنارش می‌گذاری و با داده می‌سنجی.
- grader سخت‌گیر ← شواهد را از دست می‌دهی (امتناع اشتباه)؛ grader سهل‌گیر ← جواب غلط. هر دو خطا را بشمار (گام ۴).

## گام بعد
[گام ۳: بازنویسی و حلقه‌ی اصلاحی](#)


In [ ]:
"""Step 2: grade retrieved chunks: relevant or not."""
from __future__ import annotations

import re

STOP = {"the", "a", "an", "is", "are", "do", "does", "i", "you", "my", "to", "of", "and", "or", "in", "on", "for", "can", "how",
        "what", "it", "we", "your", "with", "if", "be", "at", "by", "from", "this", "that", "when", "where", "there", "much", "many", "get", "have", "long"}
SYN = {"send back": "return", "money back": "refund", "broken": "damage defect", "couch": "sofa"}


def expand(text: str) -> str:
    text = text.lower()
    for a, b in SYN.items():
        text = text.replace(a, b)
    return text


def content_words(text: str) -> set[str]:
    # TODO 1: lowercase, words/numbers only, drop STOP and one-letter words, strip one trailing "s" from words longer than 3 letters
    raise NotImplementedError("TODO 1: content_words")


def grade(question: str, chunk: dict, min_ratio: float = 0.5) -> bool:
    # TODO 2: q = content_words(expand(question)); c = content_words(chunk heading + " " + chunk text)
    #         relevant when at least min_ratio of the question's content words are in the chunk: len(q & c) / len(q) >= min_ratio
    #         an empty question is never relevant
    raise NotImplementedError("TODO 2: grade")


if __name__ == "__main__":
    from learn.data import faq_chunks, faq_questions
    chunks = faq_chunks()
    for q in faq_questions():
        kept = [c["id"] for c in chunks if grade(q["question"], c)]
        print(f"{q['question'][:42]:42s} expected={q['relevant'] or '(none)':9s} kept={kept}")


In [ ]:
from learn.notebook import check_cell
check_cell('p22_agentic_rag', 'step02_grader', globals())

# گام ۳: بازنویسی query و حلقه‌ی اصلاحی (corrective loop)

## هدف این گام
وقتی جست‌وجوی اول شاهد مرتبط پیدا نکرد، سیستم **query را عوض می‌کند و دوباره امتحان می‌کند**، ولی تعداد دفعات محدود است. این گام دو تابع می‌نویسد: `rewrite` و `agentic_search`.

## پیش‌نیاز
گام ۲ (grader). برای تمرین یک نسخه‌ی آماده‌ی `grade` در فایل هست تا به پاسخ گام ۲ وابسته نباشی.

## مفهوم
کاربر «how long do I have to send back a sofa» می‌پرسد. متن بازگشت از «return» حرف می‌زند، نه «send back». دلیل شکست جست‌وجوی اول **واژگان** است، نه نبود پاسخ. راه حل: query دوم با کلمه‌های دیگر (`expand` عبارت `send back` را به `return` تبدیل می‌کند).

**حلقه‌ی اصلاحی:**
```
queries = [question]
repeat (حداکثر max_rounds بار):
    برای هر query که قبلاً امتحان نشده: جست‌وجو کن، با grade فیلتر کن، به relevant اضافه کن
    اگر len(relevant) >= min_relevant: توقف
    queries = rewrite(question, tried)   # query های تازه، هیچ‌کدام تکراری نباشد
    اگر queries خالی بود: توقف (ایده‌ای نمانده)
```
**سقف دور (`max_rounds`) اجباری است.** حلقه‌ی بی‌سقف هزینه و تأخیر را بی‌نهایت می‌کند؛ این یک باگ است نه یک ویژگی.

`rewrite` هم در این گام قاعده‌ای است (در گام ۵ نسخه‌ی مدل): خروجی‌ها به ترتیب:
1. `expand(question)` (مترادف‌ها باز شده)،
2. فقط کلمه‌های محتوایی سؤال (بدون کلمه‌های بی‌اهمیت)،
3. هر کلمه‌ی محتوایی به‌تنهایی (برای سؤال‌های سخت).
فقط آن‌هایی که قبلاً در `tried` نبوده‌اند.

## تمرین کد
فایل پاسخ: `work/p22_agentic_rag/step03_corrective_loop/answer.py`

1. `rewrite(question, tried)` : لیست query های تازه (بالا).
2. `agentic_search(question, search, grade, rewrite, max_rounds=3, min_relevant=1)` :
   - `search(query)` لیست تکه‌ها (دیکشنری با `id`) برمی‌گرداند.
   - `grade(question, chunk)` بولی.
   - خروجی: `{"chunks": [...], "rounds": تعداد دور انجام‌شده, "queries": [همه‌ی query های امتحان‌شده به ترتیب]}`.
   - تکه‌ی تکراری (همان `id`) فقط یک بار می‌آید.

## اجرا کن
```bash
python -m learn check p22_agentic_rag 3
python work/p22_agentic_rag/step03_corrective_loop/answer.py     # سؤال sofa را ببین
```

## خروجی مورد انتظار
شش ✓. در اجرای دستی، «how long do I have to send back a sofa» در دور اول چیزی نمی‌یابد و بعد از بازنویسی تکه‌ی بازگشت پیدا می‌شود (`rounds=2`)؛ سؤال‌های بی‌جواب بعد از ۳ دور با `chunks=[]` تمام می‌شوند.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| حلقه تمام نمی‌شود | `max_rounds` را نشمرده‌ای، یا همان query ها را دوباره می‌فرستی (`tried` را نگه‌ندار) |
| `rounds` همیشه برابر `max_rounds` است | بعد از یافتن شاهد کافی `break` نکرده‌ای |
| یک تکه دوبار در خروجی است | دیکشنری بر اساس `id` استفاده نکرده‌ای |

## دام‌ها
- بازنویسی بی‌هدف ممکن است سؤال را از معنای اصلی دور کند. grade همیشه با **سؤال اصلی** انجام می‌شود، نه query بازنویسی‌شده.
- هر دور = یک یا چند جست‌وجو (و در نسخه‌ی مدل، یک فراخوانی LLM). `rounds` و `queries` را لاگ کن تا هزینه را ببینی.

## گام بعد
[گام ۴: راستی‌آزمایی و امتناع](#)


In [ ]:
"""Step 3: rewrite the query and retry, within a round budget."""
from __future__ import annotations

import re

STOP = {"the", "a", "an", "is", "are", "do", "does", "i", "you", "my", "to", "of", "and", "or", "in", "on", "for", "can", "how",
        "what", "it", "we", "your", "with", "if", "be", "at", "by", "from", "this", "that", "when", "where", "there", "much", "many",
        "get", "have", "long"}
SYN = {"send back": "return", "money back": "refund", "broken": "damage defect", "couch": "sofa"}


def expand(text: str) -> str:
    text = text.lower()
    for a, b in SYN.items():
        text = text.replace(a, b)
    return text


def words(text: str) -> list[str]:
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP and len(w) > 1]


def rewrite(question: str, tried: list[str]) -> list[str]:
    # TODO 1: candidates in this order: expand(question); the content words joined by spaces; each content word alone.
    #         Return only candidates that are not in `tried` (and not duplicates of each other).
    raise NotImplementedError("TODO 1: rewrite")


def agentic_search(question: str, search, grade, rewrite, max_rounds: int = 3, min_relevant: int = 1) -> dict:
    """search(query) -> list of chunk dicts with "id"; grade(question, chunk) -> bool; rewrite(question, tried) -> [queries]."""
    # TODO 2: queries = [question]; tried = []; relevant = {} (by chunk id); rounds = 0
    # TODO 3: while queries and rounds < max_rounds: rounds += 1; for each query: tried.append(query), search it, keep the
    #         chunks that grade(question, chunk) accepts; stop when len(relevant) >= min_relevant; else queries = rewrite(question, tried)
    # TODO 4: return {"chunks": list(relevant.values()), "rounds": rounds, "queries": tried}
    raise NotImplementedError("TODO 2-4: the corrective loop")


if __name__ == "__main__":
    from learn.data import faq_chunks, faq_questions, retrieve, rule_grade
    chunks = faq_chunks()
    for q in faq_questions():
        out = agentic_search(q["question"], lambda s: retrieve(s, chunks, k=3), rule_grade, rewrite)
        print(f"{q['question'][:42]:42s} rounds={out['rounds']} found={[c['id'] for c in out['chunks']]}")


In [ ]:
from learn.notebook import check_cell
check_cell('p22_agentic_rag', 'step03_corrective_loop', globals())

# گام ۴: راستی‌آزمایی (verify) و حق «نمی‌دانم» (abstain)

## هدف این گام
همه‌ی قطعه‌های گام‌های قبل را در یک عامل کامل (`run_agent`) سوار می‌کنی، پاسخ را قبل از تحویل **راستی‌آزمایی** می‌کنی و اگر شاهد کافی نیست **امتناع** می‌کنی. بعد با `report` دو نوع خطا را می‌شماری: جواب غلط به سؤال بی‌جواب، و امتناع به سؤالی که جواب داشت.

## پیش‌نیاز
گام‌های ۱ تا ۳ مفهوماً. اجزا (`route`، `grade`، `agentic_search`) در فایل شروع **آماده** داده شده‌اند تا به پاسخ گام‌های قبل وابسته نباشی.

## مفهوم
حتی با grader و بازنویسی، ممکن است تکه‌ی «به‌ظاهر مرتبط» پیدا شود و پاسخ چیزی بگوید که تکه نمی‌گوید. **Verify** سومین دفاع است: «آیا هر ادعای پاسخ در شواهد هست؟». در این گام verify قاعده‌ای است: هر کلمه‌ی محتوایی پاسخ (غیر از کلمه‌های ربط) باید در متن شواهد باشد. این برای پاسخ‌های **استخراجی** (جمله‌ای از متن) درست است؛ برای پاسخ‌های نوشته‌ی مدل، در گام ۵ verify با مدل می‌آید.

**امتناع** در سه حالت اتفاق می‌افتد:
1. هیچ شاهد مرتبطی پیدا نشد (بعد از بازنویسی‌ها).
2. پاسخ ساخته شد ولی verify رد کرد.
3. پاسخ ارجاع معتبر نداشت.

خروجی `run_agent`: `{"answer": متن یا None, "citations": [...], "route": ..., "rounds": ..., "abstained": bool}`.

## تمرین کد
فایل پاسخ: `work/p22_agentic_rag/step04_verify_abstain/answer.py`

1. `verify(answer, chunks)` : درست اگر همه‌ی کلمه‌های محتوایی `answer` در مجموعه‌ی کلمه‌های متن همه‌ی `chunks` باشند (با همان ریشه‌یابی ساده). پاسخ خالی ← نادرست.
2. `run_agent(question, search, grade, rewrite, answer_fn, direct_fn=...)` :
   - `route(question)` (آماده) → اگر `direct`، با `direct_fn(question)` جواب بده (بدون جست‌وجو، `rounds=0`).
   - وگرنه `agentic_search`. اگر `chunks` خالی ← امتناع.
   - `answer_fn(question, chunks)` یک `(متن, [شناسه‌ها])` برمی‌گرداند. ارجاع‌ها را به شناسه‌های واقعی `chunks` محدود کن. اگر ارجاعی نماند یا `verify` رد کرد ← امتناع.
3. `report(outcomes, questions)` : `outcomes` لیست خروجی `run_agent` برای هر سؤال (هم‌ترتیب با `questions` دیکشنری‌های `answerable` = `yes`/`no`/`direct`) و خروجی دیکشنری:
   - `answered_correctly`: سؤال `yes` که جواب گرفت،
   - `abstained_wrongly`: سؤال `yes` که امتناع شد (**امتناع اشتباه**)،
   - `abstained_correctly`: سؤال `no` که امتناع شد،
   - `answered_unanswerable`: سؤال `no` که جواب داده شد (**خطرناک‌ترین**)،
   - `direct_ok`: سؤال `direct` که مستقیم جواب گرفت.

## اجرا کن
```bash
python -m learn check p22_agentic_rag 4
python work/p22_agentic_rag/step04_verify_abstain/answer.py    # گزارش روی ۱۵ سؤال FAQ
```

## خروجی مورد انتظار
هفت ✓. در اجرای دستی، گزارش باید `answered_unanswerable` برابر ۰ یا خیلی کم نشان دهد. اگر ۰ نشد، نگاه کن کدام سؤال بی‌جواب رد شده: معمولاً یک تکه‌ی همپوشان یا verify سهل‌گیر.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| پاسخی با ارجاع ساختگی پذیرفته شد | ارجاع‌ها را با `{c["id"] for c in chunks}` فیلتر نکردی |
| سلام به جست‌وجو رفت | شاخه‌ی `direct` را قبل از `agentic_search` نگذاشتی |
| `verify` پاسخ درست را رد می‌کند | ریشه‌یابی ساده (`s`) را نگذاشتی؛ `days` ≠ `day` |

## دام‌ها
- verify قاعده‌ای **سهل‌گیر نسبت به معنا** است: جمله‌ی «you have 90 days» ممکن است کلمه‌های متن را داشته باشد ولی عدد را عوض کند! عددها را حتماً جدا بررسی کن (تمرین اضافه).
- امتناع بی‌حساب هم شکست است: عامل بی‌فایده می‌شود. همیشه `abstained_wrongly` را کنار `answered_unanswerable` گزارش کن.

## تمرین اضافه
verify را طوری تغییر بده که **هر عدد** پاسخ باید دقیقاً در شواهد باشد. با پاسخ «You have 90 days [returns#0]» امتحان کن.

## گام بعد
[گام ۵: اجزای مبتنی بر مدل زبانی](#)


In [ ]:
"""Step 4: the full agentic loop with verification and abstention (route, grade, search helpers are given)."""
from __future__ import annotations

import re

STOP = {"the", "a", "an", "is", "are", "do", "does", "i", "you", "my", "to", "of", "and", "or", "in", "on", "for", "can", "how",
        "what", "it", "we", "your", "with", "if", "be", "at", "by", "from", "this", "that", "when", "where", "there", "much", "many",
        "get", "have", "long", "within", "will"}


def cw(text: str) -> set[str]:
    """content words with a simple plural strip"""
    return {w[:-1] if len(w) > 3 and w.endswith("s") else w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP and len(w) > 1}


def route(question: str) -> str:                                     # step 1 (given)
    q = question.strip().lower().rstrip("!.? ").strip()
    return "direct" if q in {"hello", "hi", "hey", "thanks", "thank you", "bye"} else "retrieve"


def agentic_search(question, search, grade, rewrite, max_rounds=3, min_relevant=1) -> dict:   # step 3 (given)
    queries, tried, relevant, rounds = [question], [], {}, 0
    while queries and rounds < max_rounds:
        rounds += 1
        for q in queries:
            tried.append(q)
            for c in search(q):
                if grade(question, c):
                    relevant.setdefault(c["id"], c)
        if len(relevant) >= min_relevant:
            break
        queries = [x for x in rewrite(question, tried) if x not in tried]
    return {"chunks": list(relevant.values()), "rounds": rounds, "queries": tried}


def verify(answer: str, chunks: list[dict]) -> bool:
    # TODO 1: every content word of the answer (cw) must occur in the content words of all chunk texts together;
    #         an empty answer is never verified. Citation brackets like [returns#0] must NOT count as answer words.
    raise NotImplementedError("TODO 1: verify")


def run_agent(question: str, search, grade, rewrite, answer_fn, direct_fn=lambda q: "Hello! How can I help?") -> dict:
    # TODO 2: direct route -> {"answer": direct_fn(question), "citations": [], "route": "direct", "rounds": 0, "abstained": False}
    # TODO 3: found = agentic_search(...); no chunks -> abstain (answer None)
    # TODO 4: text, cites = answer_fn(question, chunks); keep only cites that are ids of the chunks; no cite left or
    #         not verify(text, chunks) -> abstain; otherwise return the answer. Always include route and rounds.
    raise NotImplementedError("TODO 2-4: run_agent")


def report(outcomes: list[dict], questions: list[dict]) -> dict:
    # TODO 5: count the five categories described in the lesson
    raise NotImplementedError("TODO 5: report")


if __name__ == "__main__":
    from learn.data import faq_chunks, faq_questions, retrieve, rule_grade
    chunks = faq_chunks()

    def rewrite(q, tried):
        return [x for x in [" ".join(sorted(cw(q)))] if x not in tried]

    def answer_fn(q, cs):                         # extractive: the best sentence of the best chunk, cited
        best = max(cs, key=lambda c: len(cw(q) & cw(c["text"])))
        return f"{best['text']} [{best['id']}]", [best["id"]]

    qs = faq_questions()
    outs = [run_agent(q["question"], lambda s: retrieve(s, chunks, k=3), rule_grade, rewrite, answer_fn) for q in qs]
    for q, o in zip(qs, outs):
        print(f"{q['question'][:42]:42s} {'ABSTAIN' if o['abstained'] else o['answer'][:50]}")
    print(report(outs, qs))


In [ ]:
from learn.notebook import check_cell
check_cell('p22_agentic_rag', 'step04_verify_abstain', globals())

# گام ۵: اجزای مبتنی بر مدل زبانی (Ollama / Claude / OpenCode)

## هدف این گام
قاعده‌های گام‌های قبل را با **مدل زبانی** عوض می‌کنی: مسیریاب، grader، بازنویس و verifier. کدهای آماده‌ی `labkit.agentic_rag` را به یک مدل وصل می‌کنی و رفتار دو عامل (قاعده‌ای و مدلی) را روی همان سؤال‌ها مقایسه می‌کنی.

## پیش‌نیاز
گام‌های ۱ تا ۴. برای اجرای واقعی یکی از ارائه‌دهنده‌ها (Ollama، Claude، OpenCode) طبق [00_setup](#). بدون آن، مدل `offline` اجرا می‌شود که فقط شکل کار را نشان می‌دهد.

## مفهوم
قاعده‌های ما سریع و قابل‌تست‌اند ولی معنا را نمی‌فهمند: «bank transfer» و «return transfer» برایشان یکی است. یک **مدل زبانی** می‌تواند بپرسد «آیا این متن به این سؤال کمک می‌کند؟» و جواب ساخت‌یافته بدهد (`{"relevant": true}`). بهای آن: هر تکه یک فراخوانی (هزینه، تأخیر) و احتمال خطای خود مدل.

**اصل مهندسی:** مدل را **پشت یک رابط ثابت** بگذار: `llm.chat(messages)` برای متن و `llm.json(messages, schema)` برای خروجی ساخت‌یافته. بعد Ollama، Claude یا OpenCode فقط یک تنظیم است و **تست‌ها بدون مدل واقعی** (با مدل ساختگی) اجرا می‌شوند. دقیقاً همین کار را در `support/providers.py` کرده‌ایم.

اجزای آماده در `labkit.agentic_rag`:

| تابع | مدل چه می‌گوید | schema |
|---|---|---|
| `llm_router(llm)` | `direct` / `retrieve` / `multi_hop` | `{"route": enum}` |
| `llm_grader(llm)` | آیا تکه مرتبط است | `{"relevant": bool}` |
| `llm_rewriter(llm)` | query های تازه | `{"queries": [str]}` |
| `llm_verifier(llm)` | آیا پاسخ از منابع پشتیبانی می‌شود | `{"supported": bool, ...}` |

کلاس `AgenticRAG` همان حلقه‌ی گام‌های ۱ تا ۴ را با این اجزا اجرا می‌کند.

## تمرین کد
فایل پاسخ: `work/p22_agentic_rag/step05_llm_components/answer.py`

1. `to_chunks(dicts)` : دیکشنری‌های ما (`id`, `doc_id`, `heading`, `text`) را به `labkit.chunking.Chunk(chunk_id, doc_id, text, meta)` تبدیل کن؛ `heading` در `meta["heading"]` می‌رود.
2. `make_answer(llm)` : تابعی `(question, chunks) -> (text, cites)` بساز که prompt را از `Chunk`ها می‌سازد (`[chunk_id] text`)، `llm.chat(...).content` را می‌گیرد و ارجاع‌های `[...]` را که شناسه‌ی واقعی‌اند برمی‌گرداند.
3. `make_agent(search, llm, max_rounds=3)` : یک `AgenticRAG` با `grade`, `rewrite`, `verify`, `route` مبتنی بر `llm`، `answer=make_answer(llm)` و `direct` که با `llm.chat` جواب می‌دهد.

## اجرا کن
```bash
python -m learn check p22_agentic_rag 5
```
مقایسه‌ی واقعی (فایل پاسخ را بعد از قبول شدن اجرا کن):
```powershell
# پیش‌فرض: offline (بدون مدل)
python work\p22_agentic_rag\step05_llm_components\answer.py
# Ollama
$env:LEARN_PROVIDER="ollama"; $env:LEARN_MODEL="qwen2.5:7b"; python work\p22_agentic_rag\step05_llm_components\answer.py
# Claude (کلید API در .env؛ سقف هزینه را بگذار!)
$env:LEARN_PROVIDER="anthropic"; $env:LEARN_MODEL="claude-sonnet-5-5"; $env:LEARN_BUDGET_USD="0.5"
python work\p22_agentic_rag\step05_llm_components\answer.py
# OpenCode
$env:LEARN_PROVIDER="opencode"; $env:LEARN_MODEL="anthropic/claude-sonnet-5-5"; python work\p22_agentic_rag\step05_llm_components\answer.py
```
(Linux/macOS: `export X=y`.)

## خروجی مورد انتظار
تست‌ها شش ✓. اجرای مقایسه دو ردیف گزارش چاپ می‌کند (`rules` و `llm`) با همان پنج شمارنده‌ی گام ۴ و تعداد فراخوانی/تأخیر. با `offline`، مدل «همه‌چیز مرتبط است» می‌گوید و `answered_unanswerable` بالا می‌رود: این **درس اصلی** است؛ خروجی بدون مدل واقعی قابل‌اعتماد نیست و همیشه باید اندازه گرفت.

با مدل واقعی معمولاً: `answered_unanswerable` کمتر یا برابر قاعده‌ها و `abstained_wrongly` کمتر؛ ولی هزینه و زمان چند برابر. اگر مدل کوچک (۷B) از قاعده‌ها بدتر شد، تعجب نکن؛ گزارش کن.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| `KeyError: 'relevant'` | مدل JSON با شکل دیگر داد؛ prompt/schema را سخت‌تر کن یا مدل بهتر بگذار |
| `ANTHROPIC_API_KEY is not set` | کلید در `.env` نیست (اشتراک Claude.ai کلید نیست) |
| `BudgetExceeded` | سقف `LEARN_BUDGET_USD` تمام شد؛ این رفتار درست است |
| `opencode failed` | OpenCode نصب نیست یا login نکرده‌ای |
| خیلی کند | هر تکه یک فراخوانی است؛ `k` را کم کن یا مدل کوچک‌تر/سریع‌تر بگذار |

## دام‌ها
- **مدل به‌عنوان قاضی خودش سوگیری دارد.** اگر همان مدل جواب می‌دهد و همان verify می‌کند، خطاها همبسته‌اند. مدل verify را متفاوت یا قوی‌تر بگذار و قبل از اعتماد، با ۳۰ نمونه‌ی برچسب‌دار خودت بسنج (در پروژه‌ی اصلی: kappa).
- هزینه را بشمار: تعداد فراخوانی هر سؤال = `1 (route) + تعداد تکه‌ها (grade) + rewrite + 1 (answer) + 1 (verify)`. با سقف دور محدود می‌ماند.
- داده‌ی حساس را به مدل ابری نفرست.

## جمع‌بندی p22
عاملی داری که مسیر را انتخاب می‌کند، شواهد را نمره می‌دهد، خودش دوباره امتحان می‌کند، پاسخ را چک می‌کند و حق «نمی‌دانم» دارد، به‌علاوه‌ی راه اندازه‌گیری هر تصمیمش. قدم بعد: نسخه‌ی کامل با داده‌ی بزرگ‌تر (`python -m support.panel.presets scaffold p22_agentic_rag`)، ارزیابی با bootstrap CI، و مقایسه‌ی مدل‌ها در [m01_compare_models](#).


In [ ]:
"""Step 5: model-backed components (labkit.agentic_rag) behind one llm interface."""
from __future__ import annotations

import re

from labkit.agentic_rag import AgenticRAG, llm_grader, llm_rewriter, llm_router, llm_verifier
from labkit.chunking import Chunk

SYSTEM = ("Answer ONLY from the sources. After every claim write its source id in square brackets, like [returns#0]. "
          "If the sources do not contain the answer, say you do not know.")


def to_chunks(dicts: list[dict]) -> list[Chunk]:
    # TODO 1: Chunk(chunk_id=d["id"], doc_id=d["doc_id"], text=d["text"], meta={"heading": d["heading"]}) for every dict
    raise NotImplementedError("TODO 1: to_chunks")


def make_answer(llm):
    """returns answer(question, chunks) -> (text, cited chunk ids)"""
    def answer(question: str, chunks: list[Chunk]):
        # TODO 2: messages = system SYSTEM + user "Sources:\n[chunk_id] text ...\n\nQuestion: ..."; text = llm.chat(messages).content;
        #         cites = ids inside [...] that are real chunk ids (no duplicates, in order). Return (text, cites)
        raise NotImplementedError("TODO 2: answer with citations")
    return answer


def make_agent(search, llm, max_rounds: int = 3) -> AgenticRAG:
    # TODO 3: AgenticRAG(search=search, answer=make_answer(llm), grade=llm_grader(llm), rewrite=llm_rewriter(llm),
    #         verify=llm_verifier(llm), route=llm_router(llm), direct=<function: llm.chat([...user question...]).content>, max_rounds=max_rounds)
    raise NotImplementedError("TODO 3: make_agent")


if __name__ == "__main__":
    import time

    from learn import llm as learn_llm
    from learn.data import faq_chunks, faq_questions, retrieve

    dicts = faq_chunks()
    chunks = to_chunks(dicts)
    by_id = {c.chunk_id: c for c in chunks}
    model = learn_llm.get()
    agent = make_agent(lambda q: [by_id[d["id"]] for d in retrieve(q, dicts, k=3)], model)
    counts = {"answered_correctly": 0, "abstained_wrongly": 0, "abstained_correctly": 0, "answered_unanswerable": 0}
    start = time.perf_counter()
    for q in faq_questions():
        out = agent.run(q["question"])
        if q["answerable"] == "yes":
            counts["abstained_wrongly" if out.abstained else "answered_correctly"] += 1
        else:
            counts["abstained_correctly" if out.abstained else "answered_unanswerable"] += 1
        print(f"{q['question'][:42]:42s} route={out.route:9s} rounds={out.rounds} {'ABSTAIN' if out.abstained else out.answer[:40]!r}")
    print("llm agent:", counts, f"{time.perf_counter() - start:.1f}s")
    print("Compare with the rule-based agent of step 4. Which errors changed?")


In [ ]:
from learn.notebook import check_cell
check_cell('p22_agentic_rag', 'step05_llm_components', globals())